# False Analysis Notebook (Siamese Verification)

This notebook helps you **understand errors (FP/FN)**, and inspect the hardest pairs.

**Assumptions**
- Your project structure matches `src/` as in training.
- You have a finished run directory under `outputs/<experiment_name>/` containing `best.pt` and/or `final.pt`.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# -------------------------
# Set these
# -------------------------
PROJECT_ROOT = Path.cwd().resolve().parents[0]
OUTPUTS_ROOT = PROJECT_ROOT / "outputs"
EXP_NAME = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [2]:
# -------------------------
# Resolve run_dir
# -------------------------
def pick_latest_run(outputs_root: Path) -> Path:
    cands = [p for p in outputs_root.iterdir() if p.is_dir()]
    if not cands:
        raise FileNotFoundError(f"No runs found in {outputs_root}")
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

if EXP_NAME is None:
    run_dir = pick_latest_run(OUTPUTS_ROOT)
else:
    run_dir = OUTPUTS_ROOT / EXP_NAME

run_dir


WindowsPath('D:/Projects/Facial_Recognition/outputs/siamese_verification')

In [3]:
# -------------------------
# Load config (prefer merged)
# -------------------------
import yaml
from src.utils.read_config import load_config_files

def _config_paths(config_dir: Path):
    names = [
        "paths.yaml",
        "model.yaml",
        "optim.yaml",
        "train.yaml",
        "transform.yaml",
        "logging.yaml",
        "experiment.yaml",
    ]
    return [config_dir / n for n in names if (config_dir / n).exists()]

merged_cfg_path = run_dir / "config_merged.yaml"
if merged_cfg_path.exists():
    cfg = yaml.safe_load(merged_cfg_path.read_text())
else:
    # fallback: load from repo config dir
    config_dir = PROJECT_ROOT / "src" / "config"
    cfg = load_config_files(_config_paths(config_dir))

cfg.keys()


dict_keys(['paths', 'model', 'optim', 'train', 'split', 'data', 'transform', 'logging', 'experiment'])

In [4]:
# -------------------------
# Build model exactly like training
# -------------------------
from src.model.cnn_embedder import ConvEmbeddingConfig, ConvEmbeddingNet
from src.model.head import SimilarityHeadConfig, WeightedL1Head
from src.model.siamese import SiameseNet
from src.model.init import init_weights_like_reference

def infer_hw_from_cfg(cfg: dict):
    # Keep consistent with training: prefer transform.size as square
    size = int(cfg.get("transform", {}).get("size", 105))
    return size, size

enc_cfg = ConvEmbeddingConfig(**cfg["model"]["encoder"])
_ = SimilarityHeadConfig(**cfg["model"]["head"])  # kept for completeness

encoder = ConvEmbeddingNet(enc_cfg)
head = WeightedL1Head(dim=int(enc_cfg.fc_out))
model = SiameseNet(encoder=encoder, head=head).to(device)

# init + materialize FC like training
init_weights_like_reference(model)
H, W = infer_hw_from_cfg(cfg)
C = int(enc_cfg.in_channels)
with torch.no_grad():
    dummy1 = torch.zeros((1, C, H, W), device=device, dtype=torch.float32)
    dummy2 = torch.zeros((1, C, H, W), device=device, dtype=torch.float32)
    _ = model(dummy1, dummy2)

model.eval()


SiameseNet(
  (encoder): ConvEmbeddingNet(
    (conv1): Conv2d(1, 64, kernel_size=(10, 10), stride=(1, 1))
    (conv2): Conv2d(64, 128, kernel_size=(7, 7), stride=(1, 1))
    (conv3): Conv2d(128, 128, kernel_size=(4, 4), stride=(1, 1))
    (conv4): Conv2d(128, 256, kernel_size=(4, 4), stride=(1, 1))
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (bn4): Identity()
    (act): LeakyReLU(negative_slope=0.1, inplace=True)
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (fc): Linear(in_features=9216, out_features=1024, bias=True)
    (sigmoid): Sigmoid()
  )
  (head): WeightedL1Head()
)

In [5]:
# -------------------------
# Load checkpoint (prefer final.pt if it exists)
# -------------------------
ckpt_path = run_dir / "final.pt"
if not ckpt_path.exists():
    ckpt_path = run_dir / "best.pt"
if not ckpt_path.exists():
    raise FileNotFoundError(f"No checkpoint found in {run_dir} (expected final.pt or best.pt)")

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model"], strict=True)
print("Loaded:", ckpt_path)


FileNotFoundError: No checkpoint found in D:\Projects\Facial_Recognition\outputs\siamese_verification (expected final.pt or best.pt)

In [ ]:
# -------------------------
# Build dataset/loader for TEST (or VAL)
# -------------------------
from src.data.load_pairs import parse_pairs_file
from src.data.transforms import build_transform
from torch.utils.data import DataLoader
from src.data.datasets import PairPathDataset, Pair

paths = cfg["paths"]
images_root = (PROJECT_ROOT / str(paths["images_root"])).resolve()
pairs_test_path = (PROJECT_ROOT / str(paths["pairs_test"])).resolve()

eval_transform = build_transform(cfg, train=False)

test_pairs: list[Pair] = parse_pairs_file(pairs_test_path, images_root=images_root)
print("test_pairs:", len(test_pairs))

bs = int(cfg.get("optim", {}).get("batch_size", 128))
test_ds = PairPathDataset(test_pairs, transform=eval_transform)
test_loader = DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=0, pin_memory=False)


In [ ]:
# -------------------------
# Run inference and save raw predictions
# -------------------------
from tqdm.auto import tqdm

def identity_from_path(p: Path) -> str:
    # typical datasets: images/<identity>/<img>.jpg
    return p.parent.name

records = []
model.eval()

offset = 0
with torch.no_grad():
    for x1, x2, y in tqdm(test_loader, total=len(test_loader)):
        bsz = int(y.shape[0])

        batch_pairs = test_pairs[offset : offset + bsz]
        offset += bsz

        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)
        y = y.to(device)

        p_same, e1, e2 = model(x1, x2)   # p_same: (B,)
        p_same = p_same.detach().cpu().numpy()
        y_np = y.detach().cpu().numpy().astype(int)

        # embeddings for optional later analysis
        e1_np = e1.detach().cpu().numpy()
        e2_np = e2.detach().cpu().numpy()

        for i, (p1, p2, _) in enumerate(batch_pairs):
            img1 = Path(p1)
            img2 = Path(p2)
            records.append({
                "img1": str(img1),
                "img2": str(img2),
                "id1": identity_from_path(img1),
                "id2": identity_from_path(img2),
                "y_true": int(y_np[i]),
                "score": float(p_same[i]),
                "e1": e1_np[i].tolist(),
                "e2": e2_np[i].tolist(),
            })

df = pd.DataFrame(records)
fa_dir = run_dir / "false_analysis"
fa_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(fa_dir / "raw_predictions.csv", index=False)
print("Saved:", fa_dir / "raw_predictions.csv")
df.head()


In [ ]:
# -------------------------
# Choose threshold and label errors
# -------------------------
def apply_threshold(df: pd.DataFrame, tau: float) -> pd.DataFrame:
    out = df.copy()
    out["y_pred"] = (out["score"].values >= tau).astype(int)
    out["error_type"] = "TP"
    out.loc[(out.y_true == 0) & (out.y_pred == 1), "error_type"] = "FP"
    out.loc[(out.y_true == 1) & (out.y_pred == 0), "error_type"] = "FN"
    out.loc[(out.y_true == 0) & (out.y_pred == 0), "error_type"] = "TN"
    return out

tau = 0.5   # change if you want
dfa = apply_threshold(df, tau)

acc = (dfa["y_pred"] == dfa["y_true"]).mean()
fp = (dfa["error_type"] == "FP").sum()
fn = (dfa["error_type"] == "FN").sum()
print(f"tau={tau:.3f} | test accuracy={acc:.4f} | FP={fp} | FN={fn}")


In [ ]:
# -------------------------
# Optional: find best tau by maximizing accuracy on THIS set
# (Use this only for analysis; do not report this as 'final test accuracy')
# -------------------------
taus = np.linspace(0.0, 1.0, 1001)
scores = df["score"].values
y_true = df["y_true"].values

best = (-1.0, 0.5)
for t in taus:
    y_pred = (scores >= t).astype(int)
    a = (y_pred == y_true).mean()
    if a > best[0]:
        best = (a, t)

print("best_acc_on_this_set=", best[0], "best_tau=", best[1])


In [ ]:
# -------------------------
# Score distributions (pos vs neg)
# -------------------------
pos = df[df.y_true == 1]["score"].values
neg = df[df.y_true == 0]["score"].values

plt.figure(figsize=(8,5))
plt.hist(pos, bins=50, alpha=0.6, label="Positive (same)")
plt.hist(neg, bins=50, alpha=0.6, label="Negative (different)")
plt.axvline(tau, linewidth=2)
plt.title("Score distribution")
plt.xlabel("p_same score")
plt.ylabel("count")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# -------------------------
# ROC / PR curves
# -------------------------
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

fpr, tpr, _ = roc_curve(y_true, scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
plt.plot([0,1], [0,1])
plt.title("ROC curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

prec, rec, _ = precision_recall_curve(y_true, scores)
ap = average_precision_score(y_true, scores)

plt.figure(figsize=(6,6))
plt.plot(rec, prec, label=f"AP={ap:.4f}")
plt.title("Precision-Recall curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# -------------------------
# Confusion matrix at chosen tau
# -------------------------
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(dfa["y_true"], dfa["y_pred"], labels=[0,1])
cm


In [ ]:
# -------------------------
# Top hardest False Positives / False Negatives
# -------------------------
top_k = 20

fps = dfa[dfa.error_type=="FP"].sort_values("score", ascending=False).head(top_k)
fns = dfa[dfa.error_type=="FN"].sort_values("score", ascending=True).head(top_k)

fps[["score","img1","img2","id1","id2"]].head(10), fns[["score","img1","img2","id1","id2"]].head(10)


In [ ]:
# -------------------------
# Visualize pairs (images side-by-side)
# -------------------------
from PIL import Image

def show_pairs(df_subset: pd.DataFrame, title: str, n: int = 12, max_w: int = 220):
    n = min(n, len(df_subset))
    if n == 0:
        print("No samples to show:", title)
        return

    cols = 3
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(cols*6, rows*4))
    for i in range(n):
        r = df_subset.iloc[i]
        img1 = Image.open(r["img1"]).convert("RGB")
        img2 = Image.open(r["img2"]).convert("RGB")

        # simple resize for display
        def resize(im):
            w, h = im.size
            if w > max_w:
                s = max_w / w
                im = im.resize((int(w*s), int(h*s)))
            return im

        img1 = resize(img1)
        img2 = resize(img2)

        canvas = Image.new("RGB", (img1.size[0] + img2.size[0], max(img1.size[1], img2.size[1])), (0,0,0))
        canvas.paste(img1, (0,0))
        canvas.paste(img2, (img1.size[0],0))

        ax = plt.subplot(rows, cols, i+1)
        ax.imshow(canvas)
        ax.axis("off")
        ax.set_title(f"{r['error_type']} score={r['score']:.3f}\n{r['id1']} vs {r['id2']}")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_pairs(fps, "Top False Positives (highest score, but different identity)", n=12)
show_pairs(fns, "Top False Negatives (lowest score, but same identity)", n=12)


In [ ]:
# -------------------------
# Identity-level error analysis
# -------------------------
# For FP: blame both identities; for FN: blame the shared identity (id1 == id2 usually)
dfa["pair_id"] = dfa["id1"].astype(str) + " | " + dfa["id2"].astype(str)

# Error rate by identity appearing as id1
by_id1 = dfa.groupby("id1").apply(lambda g: pd.Series({
    "n": len(g),
    "acc": (g.y_pred == g.y_true).mean(),
    "fp": (g.error_type=="FP").sum(),
    "fn": (g.error_type=="FN").sum(),
})).sort_values("acc")

by_id1.head(15)


In [ ]:
# Plot worst identities by accuracy (id1 grouping)
worst = by_id1[by_id1["n"] >= 10].head(20)  # require enough samples
plt.figure(figsize=(9,5))
plt.bar(range(len(worst)), worst["acc"].values)
plt.xticks(range(len(worst)), worst.index, rotation=70)
plt.title("Worst identities (by id1 accuracy, n>=10)")
plt.ylabel("accuracy")
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()


In [ ]:
# -------------------------
# Optional: embedding t-SNE (sample)
# -------------------------
from sklearn.manifold import TSNE

# sample to keep it fast
sample_n = min(800, len(df))
sdf = df.sample(sample_n, random_state=42).reset_index(drop=True)

E = np.stack(sdf["e1"].apply(lambda x: np.array(json.loads(x) if isinstance(x,str) else x)).values)
labels = sdf["id1"].values

tsne = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=42)
Z = tsne.fit_transform(E)

plt.figure(figsize=(7,7))
plt.scatter(Z[:,0], Z[:,1], s=8)
plt.title("t-SNE of embeddings (sample)")
plt.grid(True)
plt.tight_layout()
plt.show()
